# Regularization: geometry, paths, and noise

A reusable-plot version of the supplied regularization lecture. Each plotting
method returns a Matplotlib axes, so you can add layers and customize titles.

**Setup:** use the statwrap development checkout containing these helpers.
Install that checkout into your notebook environment with
`%pip install -e /path/to/statwrap` (or `%pip install -e .` when running from its
root), then restart the kernel. The published package may not yet contain them.

The examples use two coefficients and one response. Standardization is explicit;
intercepts are fitted and are not penalized.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, KFold
from statwrap.visualization import LossSurface, plot_l1_ball, plot_l2_ball

rng = np.random.default_rng(777)
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

## Perfect collinearity

The columns are identical. Every pair with $w_1+w_2=1$ gives the same fitted
values. The OLS marker is the minimum-norm representative, not a unique minimum.
For this example, positive-alpha Lasso also has multiple optimal coefficient
vectors; coordinate descent picks one. The arbitrary OLS representative is
therefore not joined to the Lasso path.

In [ ]:
X_equal = np.array([[0, 0], [1, 1], [2, 2]])
y_equal = np.array([100, 101, 102])
collinear = LossSurface(LinearRegression(), X_equal, y_equal)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
for kind, alpha, ax in zip(["ridge", "lasso"], [0.5, 0.1], axes):
    collinear.plot_regularization(kind, alpha, ax=ax, coefficient_range=1.6,
                                  levels=[0.02, 0.08, 0.2, 0.5, 1, 2], label_contours=False)
plt.show()

## Orthogonal predictors

Centered QR columns make orthogonality exact. Equal column norms make the MSE
contours circular. With an orthogonal design, ridge rescales the OLS coefficients
and Lasso applies soft thresholding. The 10,000-row example uses the same plotting
interface as a small dataset.

In [ ]:
n = 10_000
Z = rng.normal(size=(n, 2))
Z -= Z.mean(axis=0)
Q, _ = np.linalg.qr(Z)
X_orthogonal = np.sqrt(n) * Q
y_orthogonal = X_orthogonal @ [1, 2] + rng.normal(size=n)
orthogonal = LossSurface(LinearRegression(), X_orthogonal, y_orthogonal)
ax = orthogonal.plot(relative_loss=True, levels=[0.2, 0.5, 1, 2, 4, 8],
                     title="Orthogonal, equally scaled predictors: circular contours")
orthogonal.plot_eigenvectors(ax=ax)
plt.show()

## Imperfect collinearity: wealth and income

The response depends on both correlated predictors. Standardize once for these
geometry plots, and retain the raw data for cross-validation below. DataFrame
column names become coefficient labels.

In [ ]:
n = 100
wealth = rng.normal(size=n)
rho = 0.8
income = rho * wealth + np.sqrt(1 - rho**2) * rng.normal(size=n)
X_raw = pd.DataFrame({"Wealth coefficient": wealth, "Income coefficient": income})
y_airbnb = 5 * wealth + 5 * income + rng.normal(size=n)
X_air = pd.DataFrame(StandardScaler().fit_transform(X_raw), columns=X_raw.columns)
airbnb = LossSurface(LinearRegression(), X_air, y_airbnb)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), layout="constrained")
airbnb.plot_regularization("ridge", alpha=30, ax=axes[0], label_contours=False)
airbnb.plot_regularization("lasso", alpha=1, ax=axes[1], label_contours=False)
plt.show()

The dashed boundary uses the fitted coefficient norm, so it passes through the
selected pink point. The pink curve is the exact loss contour through that fit,
showing its contact with the constraint boundary. A radius is a coefficient budget,
not alpha.

Scikit-learn uses different objective scaling:

- **Ridge:** $\mathrm{RSS} + \alpha \|w\|_2^2$.
- **Lasso:** $\mathrm{MSE}/2 + \alpha \|w\|_1$.

The background shows unregularized MSE. Equal numerical alphas should not be
interpreted as equivalent penalties. See the official
[Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)
and [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
objective definitions.

## Build the explanation one layer at a time

Reuse `ax` to preserve the existing contours and view. This boundary is an
arbitrary budget of 9; use `plot_regularization` when the boundary must match a
particular fitted alpha.

In [ ]:
ax = airbnb.plot(coefficient_range=15, label_contours=False)
airbnb.plot_constraint(9, penalty="l1", ax=ax)
airbnb.plot_lasso_path_on_surface(ax=ax)
airbnb.plot_eigenvectors(ax=ax, length=3)
ax.set_title("Correlated predictors: L1 budget and principal directions")
plt.show()

# Direct replacements for the original hand-written helper functions:
# plot_l1_ball(9, ax)
# plot_l2_ball(3, ax)

## Coefficients as the penalty increases

OLS appears at its actual alpha of zero. The horizontal axis is linear close to
zero and logarithmic above the first positive alpha. Numerical results are also
available without creating a figure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
airbnb.plot_ridge_coef_path(ax=axes[0])
airbnb.plot_lasso_coef_path(ax=axes[1])
plt.show()

path = airbnb.regularization_path("lasso", alphas=[0.1, 0.5, 1, 2, 5])
pd.DataFrame({"alpha": path.alphas,
              "wealth coefficient": path.coefficients[:, 0],
              "income coefficient": path.coefficients[:, 1],
              "training MSE": path.mse})

## Choose alpha with cross-validation

Scaling belongs inside each training fold. The pipeline below does this and
selects alpha using validation MSE. Its final refit uses the same full-data
standardization as the geometry above. Cross-validation measures prediction
performance; the contour background is training loss.

In [ ]:
cv = GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": np.logspace(-3, 3, 50)},
    scoring="neg_mean_squared_error",
    cv=KFold(5, shuffle=True, random_state=777),
)
cv.fit(X_raw, y_airbnb)
best_alpha = cv.best_params_["ridge__alpha"]
print(f"Best ridge alpha: {best_alpha:.3g}")
print(f"Mean validation MSE: {-cv.best_score_:.3f}")
airbnb.plot_regularization("ridge", best_alpha, label_contours=False,
                           title=f"Cross-validated ridge: alpha={best_alpha:.3g}")
plt.show()

## Alternate data-generating process: one relevant predictor

Now only wealth drives the response, the predictors are more correlated, and
noise is larger. Ridge distributes weight; Lasso can put a coefficient exactly
at zero. Which predictor gets selected can be unstable when their information
overlaps strongly.

In [ ]:
rng_sparse = np.random.default_rng(31)
wealth = rng_sparse.normal(size=100)
income = 0.9 * wealth + np.sqrt(1 - 0.9**2) * rng_sparse.normal(size=100)
X_sparse = StandardScaler().fit_transform(np.column_stack([wealth, income]))
y_sparse = 10 * wealth + rng_sparse.normal(0, 10, size=100)
sparse = LossSurface(LinearRegression(), X_sparse, y_sparse,
                     feature_names=["Wealth coefficient", "Income coefficient"])

fig, axes = plt.subplots(1, 2, figsize=(13, 6), layout="constrained")
sparse.plot_regularization("ridge", 30, ax=axes[0], label_contours=False)
sparse.plot_regularization("lasso", 6, ax=axes[1], label_contours=False)
plt.show()

## Linear algebra: steep and flat directions

The centered Gram matrix divided by $n$ determines the contours. Its eigenvectors
are the principal directions; the MSE Hessian has twice its eigenvalues. Small
eigenvalues mean that moving the coefficients in that direction changes fitted
values relatively little. Arrows start at the OLS minimum.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(airbnb.gram_)
print("Eigenvalues of centered X.T @ X / n:", eigenvalues)
ax = airbnb.plot(relative_loss=True, coefficient_range=4,
                 levels=[0.2, 0.5, 1, 2, 4, 8, 16], label_contours=False)
airbnb.plot_eigenvectors(ax=ax, length=2)
ax.set_title("Principal directions of the loss contours")
plt.show()

## Added orthogonal noise: the same shape at a different height

To isolate the vertical shift, project added noise off the full design with
least squares. This avoids an explicit matrix inverse and does not require a
full-rank design. Arbitrary extra noise would usually change the coefficients;
this construction deliberately preserves them.

Use **excess MSE above OLS**, identical contour levels, and identical axis limits
to compare geometry. The titles retain the different absolute MSE floors.

In [ ]:
rng_noise = np.random.default_rng(42)
X_noise = rng_noise.normal(size=(100, 2))
X_noise -= X_noise.mean(axis=0)
y_low = X_noise @ [1, 2] + rng_noise.normal(0, 0.5, size=100)
design = np.column_stack([np.ones(100), X_noise])
noise = rng_noise.normal(0, 10, size=100)
noise_orthogonal = noise - design @ np.linalg.lstsq(design, noise, rcond=None)[0]
y_high = y_low + noise_orthogonal

surfaces = [LossSurface(LinearRegression(), X_noise, y) for y in [y_low, y_high]]
np.testing.assert_allclose(surfaces[0].w_ols_, surfaces[1].w_ols_, atol=1e-12)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
for name, s, ax in zip(["Low noise", "Added orthogonal noise"], surfaces, axes):
    s.plot_ridge_path_on_surface(
        ax=ax, relative_loss=True, levels=[0.1, 0.5, 1, 2, 4, 8],
        xlim=(-1, 3), ylim=(-0.5, 4),
    )
    ax.set_title(f"{name}: minimum MSE = {s.minimum_loss_:.2f}")
plt.show()

## Small API details to keep in view

- Existing `plot`, `plot_ridge_path_on_surface`, and `plot_lasso_path_on_surface`
  calls remain available. The old `loss_range=15` is an accepted deprecated alias
  for `coefficient_range=15`, an axis half-width in coefficient units.
- `surface_loss([w1, w2])` evaluates the displayed surface. The older
  `evaluate_loss([w1, w2])` holds the base model's fitted intercept fixed unless
  you pass an explicit intercept.
- Models with `fit_intercept=False` use an intercept fixed at zero for both
  surfaces and fitted paths.
- For 3D teaching figures, use `surface.plot("3d")` or add `relative_loss=True`.
- A fitted estimator is reused without modification. An unfitted estimator is
  cloned. A fitted Ridge/Lasso base marker need not minimize unregularized MSE.